<a href="https://colab.research.google.com/github/Lallalchambugongmarak/-Major_Project/blob/main/MajorProject_AI_Study_Companion_and_Exam_preparation_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# main.py
from fastapi import FastAPI, UploadFile
from pydantic import BaseModel
from typing import List, Dict
import datetime
import uuid

app = FastAPI(title="AI Study Companion")

# --- 1. Data Models ---
class StudentProfile(BaseModel):
    student_id: str
    name: str
    subjects: List[str]
    learning_style: str = "visual" # visual, auditory, kinesthetic
    exam_date: datetime.date
    knowledge_gaps: Dict[str, float] = {} # topic: mastery 0-1

class Question(BaseModel):
    topic: str
    difficulty: int # 1-5
    question_text: str
    solution_steps: List[str]

class ChatMessage(BaseModel):
    student_id: str
    message: str
    context: str = "study" # study, doubt, exam

# --- 2. In-memory DB for demo ---
student_db = {}
knowledge_graph = {
    "Algebra": ["Arithmetic"],
    "Calculus": ["Algebra", "Trigonometry"],
    "Physics": ["Calculus", "Algebra"]
}

# --- 3. Core Functions ---

def diagnose_knowledge(student_id: str):
    """Diagnostic conversation to map what student knows"""
    return "Let's start with a quick check. Can you solve: 2x + 3 = 11?"

def generate_study_plan(profile: StudentProfile):
    """Creates personalized schedule based on exam timeline"""
    days_left = (profile.exam_date - datetime.date.today()).days
    plan = f"You have {days_left} days. We'll focus on weak topics first: {list(profile.knowledge_gaps.keys())}"
    return plan

def socratic_tutor(question: str, student_answer: str):
    """Teaches via guiding questions instead of direct answers"""
    if "wrong" in student_answer.lower():
        return "Hmm, let's think about it. What is the first step to isolate x in 2x + 3 = 11?"
    else:
        return "Correct! Now can you explain why we subtract 3 first?"

def generate_problem(topic: str, difficulty: int):
    """Generates problem at right difficulty"""
    return Question(
        topic=topic,
        difficulty=difficulty,
        question_text=f"Solve for x: {difficulty}x + {difficulty+2} = {difficulty*5}",
        solution_steps=["Subtract constant", "Divide by coefficient"]
    )

def analyze_mistake(student_answer: str, correct_answer: str):
    """Distinguish conceptual vs careless error"""
    if len(student_answer) < 3:
        return "Careless mistake: answer too short. Let's slow down."
    else:
        return "Conceptual gap: Let's review the prerequisite topic first."

def motivation_check_in(student_id: str):
    """Encouragement and accountability"""
    return "You've studied 3 days in a row! Only 20 days to exam. Let's do 2 more problems today."

# --- 4. API Endpoints ---

@app.post("/register")
def register_student(profile: StudentProfile):
    student_db[profile.student_id] = profile
    return {"message": "Student registered", "study_plan": generate_study_plan(profile)}

@app.post("/chat")
def chat(msg: ChatMessage):
    if msg.context == "study":
        reply = socratic_tutor(msg.message, "")
    elif msg.context == "doubt":
        reply = "Got your doubt. Upload the photo and I'll walk through it step by step."
    else:
        reply = motivation_check_in(msg.student_id)
    return {"reply": reply}

@app.post("/upload-doubt")
async def upload_doubt(file: UploadFile):
    # Here you would use OCR + GPT-4V to read image and solve
    return {"steps": ["Step 1: Read the problem", "Step 2: Identify formula", "Step 3: Solve"]}

@app.post("/mock-exam")
def mock_exam(student_id: str, subject: str):
    questions = [generate_problem(subject, i) for i in range(1, 4)]
    return {"timer": "60 minutes", "questions": questions}

@app.post("/feedback")
def feedback(student_id: str, was_correct: bool):
    if not was_correct:
        return {"analysis": analyze_mistake("", ""), "next_action": "Review prerequisite"}
    return {"analysis": "Great job!", "next_action": "Increase difficulty"}

In [9]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel
import datetime
from typing import List, Dict

app = FastAPI(title="AI Study Companion")

class StudentProfile(BaseModel):
    student_id: str
    name: str
    subjects: List[str]
    exam_date: datetime.date

@app.get("/")
def home():
    return {"message": "AI Study Companion is running!"}

@app.post("/register")
def register_student(profile: StudentProfile):
    return {"message": f"Welcome {profile.name}"}

Overwriting main.py


In [ ]:
!uvicorn main:app --reload --port 8000

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [7681] using WatchFiles
INFO:     Started server process [7683]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8000)